In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
!pip install xlsxwriter
import xlsxwriter
import os
import requests
import urllib3
import pandas as pd
import time
from datetime import datetime
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


## writting in the yearly folder as yearwise data !!
# directory_path = "/content/drive/My Drive/Landscape level impact assessment/mani_sarthak/final/dataScrapping/data/yearly/"
directory_path = "/content/drive/My Drive/Landscape level impact assessment/mani_sarthak/final/dataScrapping/data/yearly/check/"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 2.8 MB/s eta 0:00:00


# Helper Functions


In [3]:
def getStates(year):
    url = 'https://indiawris.gov.in/gwlbusinessdata'

    headers = {
        'Accept': 'application/json, text/plain, */*',
        'Accept-Language': 'en-GB,en;q=0.9',
        'Access-Control-Allow-Methods': 'GET,POST',
        'Access-Control-Allow-Origin': '*',
        'Connection': 'keep-alive',
        'Content-Type': 'application/json',
        'Origin': 'https://indiawris.gov.in',
        'Referer': 'https://indiawris.gov.in/wdo/',
        'Sec-Fetch-Dest': 'empty',
        'Sec-Fetch-Mode': 'cors',
        'Sec-Fetch-Site': 'same-origin',
        'Sec-GPC': '1',
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36',
        'sec-ch-ua': '"Brave";v="117", "Not;A=Brand";v="8", "Chromium";v="117"',
        'sec-ch-ua-mobile': '?0',
        'sec-ch-ua-platform': '"macOS"',
    }



    data = {"stnVal":{"qry":f"select metadata.state_name, count(distinct(metadata.station_code)),\
     coalesce(ROUND(AVG(businessdata.level)::numeric,2), 0) from public.groundwater_station as metadata\
     INNER JOIN public.gwl_timeseries_data as businessdata on metadata.station_code = businessdata.station_code where 1=1  \
     and metadata.agency_name = \'CGWB\' and to_char(businessdata.date, \'yyyy-mm\') between \'{year}-01\' and \'{year}-12\'  \
     group by metadata.state_name order by metadata.state_name"}}

    response = requests.post(url, json=data, headers=headers, verify=False)

    states = []
    if response.status_code == 200:
        response = response.json()
        for state_name in response:
            states.append(state_name[0])
    return states

In [4]:
def getDistricts(state_name, year):
    url = 'https://indiawris.gov.in/gwlbusinessdata'

    headers = {
        'Accept': 'application/json, text/plain, */*',
        'Accept-Language': 'en-GB,en;q=0.9',
        'Access-Control-Allow-Methods': 'GET,POST',
        'Access-Control-Allow-Origin': '*',
        'Connection': 'keep-alive',
        'Content-Type': 'application/json',
        'Origin': 'https://indiawris.gov.in',
        'Referer': 'https://indiawris.gov.in/wdo/',
        'Sec-Fetch-Dest': 'empty',
        'Sec-Fetch-Mode': 'cors',
        'Sec-Fetch-Site': 'same-origin',
        'Sec-GPC': '1',
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36',
        'sec-ch-ua': '"Brave";v="117", "Not;A=Brand";v="8", "Chromium";v="117"',
        'sec-ch-ua-mobile': '?0',
        'sec-ch-ua-platform': '"macOS"',
    }


    data = {
            "stnVal": {
                "qry": f"select metadata.district_name,count(distinct(businessdata.station_code)), coalesce(ROUND(AVG(businessdata.level)::numeric,2), 0) from public.groundwater_station as metadata INNER JOIN public.gwl_timeseries_data as businessdata on metadata.station_code = businessdata.station_code where 1=1  and metadata.agency_name = 'CGWB' and metadata.state_name = '{state_name}' and to_char(businessdata.date, 'yyyy') between '{year}' and '{year}'  group by district_name"
            }
        }


    response = requests.post(url, json=data, headers=headers, verify=False)
    if response.status_code == 200:
        response = response.json()
        # print(response)
    else:
        print(state_name, "couldn't be found !")

    districts = [x[0] for x in response]
    return districts


In [5]:
def getStationsInDistrict(state_name, district, year, code):
    url = 'https://indiawris.gov.in/gwlbusinessdata'

    headers = {
        'Accept': 'application/json, text/plain, */*',
        'Accept-Language': 'en-GB,en;q=0.9',
        'Access-Control-Allow-Methods': 'GET,POST',
        'Access-Control-Allow-Origin': '*',
        'Connection': 'keep-alive',
        'Content-Type': 'application/json',
        'Origin': 'https://indiawris.gov.in',
        'Referer': 'https://indiawris.gov.in/wdo/',
        'Sec-Fetch-Dest': 'empty',
        'Sec-Fetch-Mode': 'cors',
        'Sec-Fetch-Site': 'same-origin',
        'Sec-GPC': '1',
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36',
        'sec-ch-ua': '"Brave";v="117", "Not;A=Brand";v="8", "Chromium";v="117"',
        'sec-ch-ua-mobile': '?0',
        'sec-ch-ua-platform': '"macOS"',
    }
    if code == 0:
        data = {"stnVal":{"qry":f"select metadata.station_name, metadata.station_code,coalesce(ROUND(AVG(businessdata.level)::numeric,2), 0) from public.groundwater_station as metadata INNER JOIN public.gwl_timeseries_data as businessdata on metadata.station_code = businessdata.station_code where 1=1  and metadata.agency_name = \'CGWB\' and metadata.state_name = \'{state_name}\' and lower(metadata.district_name) = lower(\'{district}\') and to_char(businessdata.date, \'yyyy\') between \'{year}\' and \'{year}\'  group by metadata.station_name, metadata.station_code"}}
    else :
        x = [(None, None), ("01", "03"), ("04", "06"), ("07", "09"), ("10", "12")]
        data = {"stnVal":{"qry":f"select metadata.station_name, metadata.station_code,coalesce(ROUND(AVG(businessdata.level)::numeric,2), 0) from public.groundwater_station as metadata INNER JOIN public.gwl_timeseries_data as businessdata on metadata.station_code = businessdata.station_code where 1=1  and metadata.agency_name = \'CGWB\' and metadata.state_name = \'{state_name}\' and lower(metadata.district_name) = lower(\'{district}\') and to_char(businessdata.date, \'yyyy-mm\') between \'{year}-{x[code][0]}\' and \'{year}-{x[code][1]}\'  group by metadata.station_name, metadata.station_code"}}

    stations = requests.post(url, json=data, headers=headers, verify=False)
    if stations.status_code == 200:
        stations = stations.json()
        # print(district, stations)
    else:
        print(state_name, "couldn't be found !")

    return stations


In [6]:
def getLatLong(station_id):

    url = 'https://arc.indiawris.gov.in/server/rest/services/NWIC/GroundwaterLevel_Stations/MapServer/0/query'

    headers = {
        'authority': 'arc.indiawris.gov.in',
        'accept': '*/*',
        'accept-language': 'en-GB,en;q=0.9',
        'origin': 'https://indiawris.gov.in',
        'referer': 'https://indiawris.gov.in/',
        'sec-ch-ua': '"Brave";v="117", "Not;A=Brand";v="8", "Chromium";v="117"',
        'sec-ch-ua-mobile': '?0',
        'sec-ch-ua-platform': '"macOS"',
        'sec-fetch-dest': 'empty',
        'sec-fetch-mode': 'cors',
        'sec-fetch-site': 'same-site',
        'sec-gpc': '1',
        'user-agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36',
    }

    params = {
        'f': 'json',
        'outFields': '*',
        'spatialRel': 'esriSpatialRelIntersects',
        'where': f'station_code = \'{station_id}\'',
    }


    response = requests.get(url, headers=headers, params=params)

    if response.status_code == 200:
        response = response.json()
        data = response
        features = data.get('features', [])
        if features:
            first_feature = features[0]
            attributes = first_feature.get('attributes', {})
            latitude = attributes.get('lat')
            longitude = attributes.get('long')
            if latitude is not None and longitude is not None:
                return latitude, longitude
    return None, None


In [7]:
def addLatLong(data):
    new_data = []
    for x in data:
        station_id = x[1]
        (lat, long) = getLatLong(station_id)
        new_data.append(x + [lat, long])
    return new_data

In [8]:
def create_excel(data, year, state, district, loc, season = None):
    columns = ["station_name", "station_id", "water_level (in m)", "lat", "long"]
    df = pd.DataFrame(data, columns = columns)
    if not os.path.exists(loc):
        os.makedirs(loc)
    if season != 0:
        excel_file = loc + f"{district}_{state}_{year}_{season}.xlsx"
    else:
        excel_file = loc + f"{district}_{state}_{year}_yearly.xlsx"
    with pd.ExcelWriter(excel_file, engine="xlsxwriter") as writer:
        # print(df)
        df.to_excel(writer, sheet_name="Sheet1", index=False)
    print("file written", loc)

In [9]:
def fetchData(year, state, district, season):
    data = getStationsInDistrict(state, district, year, season)
    data = addLatLong(data)
    # file_path = directory_path+f"{year}/{state}/{district}/"
    file_path = directory_path
    create_excel(data, year, state, district, file_path, season)

In [10]:
# fetchData(2023, 'Bihar', 'Saran', 2)

# Functions to fetch data yearwise
Please do not change below this.

In [11]:
# year to fetch from before ie 2023 2022 2021 and so on till 1994
LAST_YEAR = datetime.now().year
LAST_STATE = None
LAST_DISTRICT = None

In [12]:
log_file = directory_path + "log.txt"
progress_file = directory_path + "progress.txt"


def log_timestamp(message):
    timestamp = time.strftime("%Y-%m-%d %H:%M:%S")
    with open(log_file, "a") as log:
        log.write(f"{timestamp} - {message}\n")



In [13]:
if not os.path.exists(directory_path):
    os.makedirs(directory_path)

if os.path.isfile(progress_file):
    with open(progress_file, 'r') as f:
        last_year = int(f.readline().strip())
        last_state = f.readline().strip()
        last_district = f.readline().strip()
else:
    last_year = LAST_YEAR
    last_state = LAST_STATE
    last_district = LAST_DISTRICT

In [ ]:
log_timestamp(f"Starting from year {last_year}, state {last_state}, district {last_district}")
start_time = time.perf_counter()


for year in range(last_year, 1994, -1):
    log_timestamp(f"Started processing year {year}")
    dp_year = directory_path + str(year) + '/'

    if not os.path.exists(dp_year):
        os.makedirs(dp_year)

    states = getStates(year)

    if last_state:
        states = states[states.index(last_state):]

    for state in states:
        log_timestamp(f"Started processing state {state}")
        dp_state = dp_year + str(state) + '/'

        if not os.path.exists(dp_state):
            os.makedirs(dp_state)

        districts = getDistricts(state, year)

        if last_district:
            districts = districts[districts.index(last_district):]

        for district in districts:
            with open(progress_file, 'w') as f:
                f.write(str(year) + '\n')
                f.write(state + '\n')
                f.write(district + '\n')
            log_timestamp(f"Started processing district {district}")
            dp_district = dp_state + str(district) + '/'

            if not os.path.exists(dp_district):
                os.makedirs(dp_district)

            for season in range(5):
                data = getStationsInDistrict(state, district, year, season)
                data = addLatLong(data)
                create_excel(data, year, state, district, dp_district, season)

            log_timestamp(f"Finished processing district {district}")

        log_timestamp(f"Finished processing state {state}")
        last_district = None
    log_timestamp(f"Finished processing year {year}")
    last_state = None



log_timestamp("Script finished")
total_time = time.perf_counter() - start_time
log_timestamp(f"Total time taken: {total_time} seconds")


file written /content/drive/My Drive/Landscape level impact assessment/mani_sarthak/final/dataScrapping/data/yearly/check/2023/Andaman & Nicobar/North Middle Andaman/
file written /content/drive/My Drive/Landscape level impact assessment/mani_sarthak/final/dataScrapping/data/yearly/check/2023/Andaman & Nicobar/North Middle Andaman/
file written /content/drive/My Drive/Landscape level impact assessment/mani_sarthak/final/dataScrapping/data/yearly/check/2023/Andaman & Nicobar/North Middle Andaman/
file written /content/drive/My Drive/Landscape level impact assessment/mani_sarthak/final/dataScrapping/data/yearly/check/2023/Andaman & Nicobar/North Middle Andaman/
file written /content/drive/My Drive/Landscape level impact assessment/mani_sarthak/final/dataScrapping/data/yearly/check/2023/Andaman & Nicobar/North Middle Andaman/
file written /content/drive/My Drive/Landscape level impact assessment/mani_sarthak/final/dataScrapping/data/yearly/check/2023/Andaman & Nicobar/South Andaman/
file 